
# Solar Filament Segmentation Challenge 2026

### Project Overview

This project focuses on the automatic identification and segmentation of **solar filaments** in H-alpha solar images using Computer Vision and Deep Learning.

Solar filaments are elongated structures of relatively cool, dense plasma suspended above the Sun's surface by magnetic fields. They appear as dark, irregular structures in H-alpha observations.

The primary goal is to develop a segmentation model capable of accurately detecting and delineating individual solar filaments while preserving their complex morphological characteristics.

**Competition:** [Solar Filament Segmentation Challenge 2026](https://www.kaggle.com/competitions/filament-segmentation-2026)

**Domain:** Computer Vision | Instance Segmentation | Solar Physics

<div align="center">
    <img src="../images\inbox_3070992_77fe20a042912564b30d941836d26f70_filament_GONG_20240103224642Ch.png" width="500">
    <p><em>Figure 1: Solar filament observed in H-alpha.</em></p>
</div>



### Project Objectives

The main objective is to build a robust and efficient model for **solar filament instance segmentation**.

Specific objectives include:

- **Data Exploration:** Investigate solar images, filament annotations, and dataset characteristics.
- **Data Preprocessing:** Prepare images and transform annotations into segmentation masks.
- **Model Development:** Implement and experiment with image segmentation architectures.
- **Instance Identification:** Distinguish individual filaments, including closely positioned structures.
- **Morphological Preservation:** Accurately segment thin, elongated, and irregular filament structures.
- **Model Evaluation:** Assess segmentation performance using Panoptic Quality (PQ) and complementary metrics.
- **Generalization:** Develop a model capable of producing reliable predictions on unseen solar observations.

<div align="center">
    <img src="..\images\inbox_3070992_7b19d336400715aa6b69a895f6cee0ed_magfilo_segmentation_examples.png" width="800">
    <p><em>Figure 2: Solar filament observed in H-alpha.</em></p>
</div>


### Challenges

Solar filament segmentation presents several challenges:

- **Complex morphology:** Filaments contain thin, elongated, and irregular structures.
- **Low contrast:** Filaments may be difficult to distinguish from surrounding solar regions.
- **Image noise:** Ground-based observations contain atmospheric and instrumental artifacts.
- **Structural continuity:** Segmentation models must preserve connected filament structures.
- **Instance separation:** Nearby filaments must be identified without incorrectly merging them.


#### Import libraries

In [ ]:
import os
import sys
import json
import requests
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image ,ImageDraw

import ast

from matplotlib import pyplot as plt
from matplotlib.patches import Polygon


import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader ,Dataset
from tqdm import tqdm
from collections import defaultdict
from pycocotools import mask as mask_utils
from sklearn.model_selection import GroupShuffleSplit

import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.io import decode_image
from sklearn.model_selection import train_test_split


project_root = Path.cwd().parent
src_path = project_root / "src"

sys.path.insert(0, str(src_path))

from tools import get_image , show_image_with_segmentation



### Extract and View Datasets

In [ ]:
train_images_dir = Path("../data/train/train_images")
test_images_dir = Path("../data/test/test_images")
MAGFILO = Path("../data/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json")

In [ ]:
print("Exists:", MAGFILO.exists())
print("Content:")
print(MAGFILO.read_text(encoding="utf-8"))

In [ ]:



with open(MAGFILO, "r", encoding="utf-8") as f:
    data = json.load(f)


print(type(data))
print(data.keys())




In [ ]:
print(data["info"])

In [ ]:
print(data["licenses"])

In [ ]:
print(data['categories'])

In [ ]:
print(data['images']) , print(len(data['images']))

In [ ]:
print(data['annotations'][0]) 

In [ ]:
print(len(data['annotations']))

In [ ]:
"""
categories_df = pd.DataFrame(data["categories"])
images_df = pd.DataFrame(data["images"])
annotations_df = pd.DataFrame(data["annotations"])

"""

In [ ]:
"""

categories_df.to_csv("../data/train/categories.csv", index=False)
images_df.to_csv("../data/train/images.csv", index=False)
annotations_df.to_csv("../data/train/annotations.csv", index=False)

"""

In [ ]:
categories_df = pd.read_csv("../data/train/categories.csv")
images_df = pd.read_csv("../data/train/images.csv")
annotations_df = pd.read_csv("../data/train/annotations.csv")

In [ ]:
display(categories_df.head(10))

In [ ]:
display(images_df.head()) , len(images_df)

In [ ]:
display(annotations_df.head()) , len(annotations_df)

In [ ]:
match_counts = annotations_df['image_id'].isin(images_df['id']).value_counts()
print(match_counts)

In [ ]:
annotations_df.category_id.value_counts()

In [ ]:
images_columns = images_df.columns
images_columns

In [ ]:
annotations_columns = annotations_df.columns
annotations_columns


In [ ]:

"""

images_renamed = images_df.rename(columns={'id': 'image_id'})


solar_df = pd.merge(
    images_renamed,
    annotations_df,
    on='image_id',
    how='inner'
)


print(solar_df.columns)


"""

In [ ]:
"""
columns_reorder = ['id','image_id', 'date_captured', 'file_name'
       , 'segmentation', 'area', 'iscrowd', 'spine', 'bbox',
       'category_id','height', 'width']

solar_df = solar_df[columns_reorder]

"""

In [ ]:
# solar_df.to_csv("../data/train/solar_data.csv", index=False)

In [ ]:
solar_path = Path("../data/train/solar_data.csv")

solar_df = pd.read_csv(solar_path)

In [ ]:

solar_df["image"] = pd.Series(
    [None] * len(solar_df),
    index=solar_df.index,
    dtype=object
)

for index, row in solar_df.iterrows():
    image, image_array = get_image(row["file_name"])

    if index == 0:
        print("Before assignment:", type(image_array), image_array.dtype)

    solar_df.at[index, "image"] = image_array

print("After assignment:", type(solar_df.iloc[0]["image"]))
print("Column dtype:", solar_df["image"].dtype) 

In [ ]:
"""
solar_path = Path("../data/train/solar_data.csv")

solar_df = pd.read_csv(solar_path)

"""

In [ ]:
display(solar_df.head()) , len(solar_df)

In [ ]:
solar_columns  = solar_df.columns
solar_columns

In [ ]:
solar_df.isnull().sum()

In [ ]:
seg = solar_df.iloc[0]["segmentation"]

seg = np.array(ast.literal_eval(seg))

for polygon in seg:
    points = np.asarray(polygon).reshape(-1, 2)
    print(points)
print(len(points))

In [ ]:
solar_df.iloc[0].bbox

In [ ]:
image, image_array = get_image(solar_df.iloc[0].file_name)

plt.imshow(image_array, cmap='grey')
plt.axis('off')
plt.show()

In [ ]:
show_image_with_segmentation(df=solar_df , image_path = solar_df.iloc[0].file_name)  


In [ ]:

image_array = np.asarray(image.convert("L"))

plt.figure(figsize=(8, 4))
plt.hist(image_array.ravel(), bins=256, range=(0, 256))
plt.xlabel("Pixel intensity (0 = black, 255 = white)")
plt.ylabel("Number of pixels")
plt.title("Image intensity histogram")
plt.show()

In [ ]:
bbox = ast.literal_eval(solar_df.iloc[0].bbox)


x = int(bbox[0])
y = int(bbox[1])
w = int(bbox[2])
h = int(bbox[3])

array_inside_box = image_array[y : y + h, x : x + w]

plt.hist(array_inside_box)

### Data preprocessing

### Building Models

In [ ]:
NUM_EPOCHS = 10
BATCH_SIZE = 64
TEST_SIZE = 0.2

INIT_LR = 0.001

BASE_OUTPUT = "output"


In [ ]:

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(solar_df, groups=solar_df["image_id"])
)

train_df = solar_df.iloc[train_idx]
val_df = solar_df.iloc[val_idx]


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())


if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
class SolarDataset(Dataset):
    def __init__(self, df, augment=False):
        self.augment = augment
        self.images = []
        self.segmentations = []

        # Each image can have several rows: one for each filament.
        for image_id, rows in df.groupby("image_id", sort=False):
            self.images.append(rows["image"].iloc[0])
            self.segmentations.append(rows["segmentation"].tolist())

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Images in our dataframe are grayscale NumPy arrays.
        image = self.images[idx]
        height, width = image.shape
        mask = np.zeros((height, width), dtype=np.uint8)

        # Fill all filament polygons into one binary mask.
        for polygons in self.segmentations[idx]:
            if isinstance(polygons, str):
                polygons = ast.literal_eval(polygons)
            if polygons is None or len(polygons) == 0:
                continue

            encoded_polygons = mask_utils.frPyObjects(polygons, height, width)
            filament_mask = mask_utils.decode(mask_utils.merge(encoded_polygons))
            mask[filament_mask > 0] = 1

        # Convert to float tensors with shape [1, height, width].
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0) / 255.0
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        if self.augment:
            if torch.rand(()) < 0.5:  # Horizontal flip
                image = image.flip(-1)
                mask = mask.flip(-1)
            if torch.rand(()) < 0.5:  # Vertical flip
                image = image.flip(-2)
                mask = mask.flip(-2)

        return image, mask


In [ ]:
train_set = SolarDataset(train_df, augment=True)

val_set = SolarDataset(val_df , augment=True)


train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

In [ ]:
class Block(nn.Module):
	def __init__(self, inChannels, outChannels):
		super().__init__()

		self.conv1 = nn.Conv2d(inChannels, outChannels,kernel_size= 3, padding=1)
		self.relu1 = nn.ReLU(inplace=True)
		self.conv2 = nn.Conv2d(outChannels, outChannels,kernel_size= 3,padding=1)
		self.relu2 = nn.ReLU(inplace=True)
		
		
	def forward(self, x):
		
		return self.relu2(self.conv2(self.relu1(self.conv1(x))))

In [ ]:
class Encoder(nn.Module):

	def __init__(self, channels=(1 , 64, 128, 256 , 512, 1024)):
		super().__init__()
		
		self.encBlocks = nn.ModuleList(
			[Block(channels[i], channels[i + 1])
			 	for i in range(len(channels) - 1)])
		
		self.pool = nn.MaxPool2d(kernel_size=2 , stride=2)
		
	def forward(self, x):
		
		blockOutputs = []

		for block in self.encBlocks:
			x = block(x)
			blockOutputs.append(x)
			x = self.pool(x)
		
		return blockOutputs

In [ ]:
class Decoder(nn.Module):
    def __init__(self, channels=(1024, 512, 256, 128, 64)):
        super().__init__()
        self.upconvs = nn.ModuleList([
            nn.ConvTranspose2d(channels[i], channels[i + 1], 2, 2)
            for i in range(len(channels) - 1)
        ])
        self.dec_blocks = nn.ModuleList([
            Block(channels[i], channels[i + 1])
            for i in range(len(channels) - 1)
        ])

    def forward(self, x, encFeatures):
        for i in range(len(self.upconvs)):
            x = self.upconvs[i](x)
            skip = encFeatures[i]

            # Odd image dimensions can leave a one-pixel size difference.
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(
                    x, size=skip.shape[-2:], mode="bilinear", align_corners=False
                )

            x = torch.cat([x, skip], dim=1)
            x = self.dec_blocks[i](x)

        return x


In [ ]:
class UNet(nn.Module):
    def __init__(self, inChannels: tuple, outChannels: tuple):
        super().__init__()
        self.encoders = Encoder(inChannels)
        self.decoders = Decoder(outChannels)
        self.head = nn.Conv2d(outChannels[-1], 1, kernel_size=1)

    def forward(self, x):
        # Encoder returns features from shallowest to deepest.
        features = self.encoders(x)
        deepest = features[-1]
        skips = features[:-1][::-1]

        x = self.decoders(deepest, skips)
        return self.head(x)


In [ ]:
IN_CHANNELS =(1 , 64, 128, 256 , 512)
OUT_CHANNELS=(512 , 256 , 128, 64)


import time
model = UNet(inChannels=IN_CHANNELS ,outChannels=OUT_CHANNELS).to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=INIT_LR)

trainSteps = len(train_loader)
testSteps = len(val_loader)
# initialize a dictionary to store training history
H = {"train_loss": [], "test_loss": []}


startTime = time.time()

for e in tqdm(range(NUM_EPOCHS)):
	# set the model in training mode
	model.train()
	# initialize the total training and validation loss
	totalTrainLoss = 0
	totalTestLoss = 0
	# loop over the training set
	for (i, (x, y)) in enumerate(train_loader):
		# send the input to the device
		(x, y) = (x.to(device), y.to(device))
		# perform a forward pass and calculate the training loss
		pred = model(x)
		loss = criterion(pred, y)
		# first, zero out any previously accumulated gradients, then
		# perform backpropagation, and then update model parameters
		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
		# add the loss to the total training loss so far
		totalTrainLoss += loss.detach()
	# switch off autograd
	with torch.no_grad():
		# set the model in evaluation mode
		model.eval()
		# loop over the validation set
		for (x, y) in val_loader:
			# send the input to the device
			(x, y) = (x.to(device), y.to(device))
			# make the predictions and calculate the validation loss
			pred = model(x)
			totalTestLoss += criterion(pred, y)
	# calculate the average training and validation loss
	avgTrainLoss = totalTrainLoss / trainSteps
	avgTestLoss = totalTestLoss / testSteps
	# update our training history

	H["train_loss"].append(avgTrainLoss.cpu().detach().numpy())
	H["test_loss"].append(avgTestLoss.cpu().detach().numpy())

	# print the model training and validation information

	print("[INFO] EPOCH: {}/{}".format(e + 1, NUM_EPOCHS))
	print("Train loss: {:.6f}, Test loss: {:.4f}".format(
		avgTrainLoss, avgTestLoss))
	
# display the total time needed to perform the training

endTime = time.time()
print("[INFO] total time taken to train the model: {:.2f}s".format(
	endTime - startTime))